# Leaf Dieback Detection Model v5

## Improvements over v4:
- Enhanced Focal Loss with label smoothing
- Optimized class weights with healthy boost
- Two-phase training (frozen base + fine-tuning)
- Comprehensive per-class metrics visualization

## Dataset:
- Classes: healthy, leaf_die_back, not_cocount
- Train/Val/Test split with data augmentation for training only

**Author:** Research Team  
**Date:** January 2026

## 1. Import Libraries and Check Environment

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"NumPy Version: {np.__version__}")

ModuleNotFoundError: No module named 'tensorflow'

## 2. Configuration and Hyperparameters

In [ ]:
# Configuration
MODEL_VERSION = "v5"
MODEL_NAME = "leaf_dieback"
BASE_DIR = r"C:\Users\Tharindu Nandun\Desktop\Research\Research\ml"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed", "leaf_dieback_v1")
MODEL_DIR = os.path.join(BASE_DIR, "models", f"{MODEL_NAME}_{MODEL_VERSION}")

# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
PHASE1_EPOCHS = 15  # Frozen base training
PHASE2_EPOCHS = 35  # Fine-tuning
INITIAL_LR = 0.001
DROPOUT_RATE = 0.4
LABEL_SMOOTHING = 0.1
FOCAL_GAMMA = 2.5
FOCAL_ALPHA = 0.3

# Create model directory
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Model: {MODEL_NAME}_{MODEL_VERSION}")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Phase 1 Epochs: {PHASE1_EPOCHS}")
print(f"Phase 2 Epochs: {PHASE2_EPOCHS}")
print(f"Initial Learning Rate: {INITIAL_LR}")
print(f"Dropout Rate: {DROPOUT_RATE}")
print(f"Label Smoothing: {LABEL_SMOOTHING}")
print(f"Focal Loss Gamma: {FOCAL_GAMMA}")
print(f"Focal Loss Alpha: {FOCAL_ALPHA}")
print(f"\nData directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")

## 3. Define Focal Loss

Focal Loss helps handle class imbalance by down-weighting easy examples and focusing on hard ones.

In [ ]:
class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss for handling class imbalance.
    
    Focal Loss adds a modulating factor (1-p)^gamma to cross-entropy loss,
    focusing learning on hard misclassified examples.
    
    Args:
        gamma: Focusing parameter. Higher values focus more on hard examples.
        alpha: Weighting factor for the classes.
        label_smoothing: Label smoothing factor for regularization.
    """
    def __init__(self, gamma=2.0, alpha=0.25, label_smoothing=0.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    
    def call(self, y_true, y_pred):
        # Apply label smoothing
        if self.label_smoothing > 0:
            num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        
        # Clip predictions to prevent log(0)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
        
        # Calculate cross entropy
        cross_entropy = -y_true * tf.math.log(y_pred)
        
        # Calculate focal weight
        weight = self.alpha * y_true * tf.pow(1 - y_pred, self.gamma)
        
        # Apply focal weight to cross entropy
        focal_loss = weight * cross_entropy
        
        return tf.reduce_sum(focal_loss, axis=-1)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'gamma': self.gamma,
            'alpha': self.alpha,
            'label_smoothing': self.label_smoothing
        })
        return config

print("Focal Loss configured with:")
print(f"  - Gamma: {FOCAL_GAMMA} (focus on hard examples)")
print(f"  - Alpha: {FOCAL_ALPHA} (class weight factor)")
print(f"  - Label Smoothing: {LABEL_SMOOTHING}")

## 4. Dataset Analysis

In [ ]:
# Analyze dataset distribution
print("="*70)
print("DATASET DISTRIBUTION")
print("="*70)

splits_data = {}
for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_DIR, split)
    splits_data[split] = {}
    print(f"\n{split.upper()}:")
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            splits_data[split][cls] = count
            print(f"  {cls}: {count}")

# Visualize distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#2ecc71', '#e74c3c', '#3498db']

for idx, split in enumerate(['train', 'val', 'test']):
    classes = list(splits_data[split].keys())
    counts = list(splits_data[split].values())
    axes[idx].bar(classes, counts, color=colors)
    axes[idx].set_title(f'{split.upper()} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)
    for i, v in enumerate(counts):
        axes[idx].text(i, v + 10, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Load and Prepare Data

In [ ]:
# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

# No augmentation for validation/test
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
train_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Get class information
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print(f"\n{'='*70}")
print("DATASET SUMMARY")
print("="*70)
print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples: {test_generator.samples}")

## 6. Visualize Sample Images

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for row, cls in enumerate(class_names):
    cls_path = os.path.join(DATA_DIR, 'train', cls)
    images = os.listdir(cls_path)[:4]
    for col, img_name in enumerate(images):
        img_path = os.path.join(cls_path, img_name)
        img = plt.imread(img_path)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=12, fontweight='bold')

plt.suptitle('Sample Images from Each Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Sample images saved to: sample_images.png")

## 7. Calculate Class Weights

In [ ]:
# Calculate class weights to handle imbalance
train_labels = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights_array))

# Extra boost for healthy class (common problem in disease detection)
healthy_idx = list(train_generator.class_indices.keys()).index('healthy') if 'healthy' in train_generator.class_indices else None
if healthy_idx is not None:
    class_weights[healthy_idx] *= 2.0  # Double the weight for healthy

print("="*70)
print("CLASS WEIGHTS")
print("="*70)
print("Class weights (to handle imbalance):")
for cls, idx in train_generator.class_indices.items():
    boost = " (2x boost)" if cls == 'healthy' else ""
    print(f"  {cls}: {class_weights[idx]:.4f}{boost}")

## 8. Build Model Architecture

Using MobileNetV2 as the base model with a custom classification head.

In [ ]:
def build_model(num_classes, dropout_rate=0.4):
    """Build MobileNetV2 model with custom classification head."""
    
    # Load pre-trained MobileNetV2 (without top layers)
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build custom classification head
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate * 0.75)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    
    return model, base_model

# Build model
print("Building model...")
model, base_model = build_model(num_classes, DROPOUT_RATE)

print("="*70)
print("MODEL ARCHITECTURE - MobileNetV2")
print("="*70)
print(f"Model: MobileNetV2 + Custom Head")
print(f"Total params: {model.count_params():,}")
trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
print(f"Trainable params: {trainable:,}")
print(f"Non-trainable params: {non_trainable:,}")

model.summary()

## 9. Phase 1: Train with Frozen Base

In [ ]:
# Compile model for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss=FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

# Callbacks for Phase 1
callbacks_phase1 = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'phase1_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("="*70)
print("PHASE 1: TRAINING WITH FROZEN BASE")
print("="*70)
print("Training classifier head only (base frozen)...\n")

# Train Phase 1
history_phase1 = model.fit(
    train_generator,
    epochs=PHASE1_EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=1
)

print(f"\nPhase 1 Complete!")
print(f"Best Validation Accuracy: {max(history_phase1.history['val_accuracy'])*100:.2f}%")

## 10. Phase 2: Fine-tuning

Unfreeze top layers of base model for fine-tuning.

In [ ]:
# Unfreeze top layers for fine-tuning
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30  # Unfreeze last 30 layers

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR/10),
    loss=FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

# Callbacks for Phase 2
callbacks_phase2 = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

print("="*70)
print("PHASE 2: FINE-TUNING")
print("="*70)
print(f"Unfreezing top 30 layers for fine-tuning...")
print(f"Total layers: {len(base_model.layers)}")
print(f"Trainable layers: {len(base_model.layers) - fine_tune_at}\n")

# Train Phase 2
history_phase2 = model.fit(
    train_generator,
    epochs=PHASE2_EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1
)

print(f"\nPhase 2 Complete!")
print(f"Best Validation Accuracy: {max(history_phase2.history['val_accuracy'])*100:.2f}%")

## 11. Training History Visualization

In [ ]:
# Combine histories
combined_history = {
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss'],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(combined_history['accuracy'], label='Train Accuracy', linewidth=2, color='#3498db')
axes[0].plot(combined_history['val_accuracy'], label='Val Accuracy', linewidth=2, color='#e74c3c')
axes[0].axvline(x=len(history_phase1.history['accuracy'])-1, color='gray', linestyle='--', 
                label='Fine-tuning Start', alpha=0.7)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.6, 1.0])

# Loss plot
axes[1].plot(combined_history['loss'], label='Train Loss', linewidth=2, color='#3498db')
axes[1].plot(combined_history['val_loss'], label='Val Loss', linewidth=2, color='#e74c3c')
axes[1].axvline(x=len(history_phase1.history['loss'])-1, color='gray', linestyle='--', 
                label='Fine-tuning Start', alpha=0.7)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Training History - Leaf Dieback {MODEL_VERSION}', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Training history saved to: training_history.png")

## 12. Evaluate on Test Set

In [ ]:
# Load best model
print("="*70)
print("EVALUATION ON TEST SET")
print("="*70)

best_model_path = os.path.join(MODEL_DIR, 'best_model.keras')
if os.path.exists(best_model_path):
    model = keras.models.load_model(
        best_model_path, 
        custom_objects={'FocalLoss': FocalLoss}
    )
    print(f"Loaded best model from: {best_model_path}")

# Evaluate on test set
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

## 13. Classification Report (Class-wise Metrics)

In [ ]:
# Get predictions
test_generator.reset()
predictions = model.predict(test_generator, verbose=0)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Classification Report
print("="*70)
print("CLASSIFICATION REPORT (Class-wise Metrics)")
print("="*70)
print()
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Detailed metrics
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

print("\n" + "="*70)
print("DETAILED CLASS-WISE METRICS")
print("="*70)
print(f"{'Class':<16} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-"*63)
for class_name in class_names:
    metrics = report_dict[class_name]
    print(f"{class_name:<16} {metrics['precision']:>10.4f} {metrics['recall']:>10.4f} {metrics['f1-score']:>10.4f} {int(metrics['support']):>10}")
print("-"*63)
print(f"{'Macro Avg':<16} {report_dict['macro avg']['precision']:>10.4f} {report_dict['macro avg']['recall']:>10.4f} {report_dict['macro avg']['f1-score']:>10.4f}")

# Summary
macro_f1 = report_dict['macro avg']['f1-score']
print(f"\n{'='*70}")
print("SUMMARY METRICS")
print("="*70)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Macro F1-Score: {macro_f1*100:.2f}%")
print(f"Difference (Acc - F1): {abs(test_accuracy - macro_f1)*100:.2f}%")

## 14. Confusion Matrix

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'size': 14})
plt.title(f'Confusion Matrix - Leaf Dieback {MODEL_VERSION}\nTest Accuracy: {test_accuracy*100:.2f}%', 
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to: confusion_matrix.png")

print("\nConfusion Matrix:")
print(cm)

## 15. Per-Class Metrics Visualization

In [ ]:
# Per-class metrics bar chart
metrics_data = {
    'Precision': [report_dict[cls]['precision'] for cls in class_names],
    'Recall': [report_dict[cls]['recall'] for cls in class_names],
    'F1-Score': [report_dict[cls]['f1-score'] for cls in class_names]
}

x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, metrics_data['Precision'], width, label='Precision', color='#3498db')
bars2 = ax.bar(x, metrics_data['Recall'], width, label='Recall', color='#2ecc71')
bars3 = ax.bar(x + width, metrics_data['F1-Score'], width, label='F1-Score', color='#e74c3c')

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Metrics (Precision, Recall, F1-Score)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'per_class_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Per-class metrics saved to: per_class_metrics.png")

## 16. Supervisor Requirements Check

In [ ]:
print("="*70)
print("SUPERVISOR REQUIREMENTS CHECK")
print("="*70)

all_good = True
results = []

for cls in class_names:
    p = report_dict[cls]['precision']
    r = report_dict[cls]['recall']
    f1 = report_dict[cls]['f1-score']
    diff = max(p, r, f1) - min(p, r, f1)
    
    status = "✓ PASS" if diff < 0.15 else "✗ FAIL"
    if diff >= 0.15:
        all_good = False
    
    results.append({
        'class': cls,
        'precision': p,
        'recall': r,
        'f1': f1,
        'diff': diff,
        'status': status
    })
    
    print(f"\n{cls}:")
    print(f"  Precision: {p:.4f}")
    print(f"  Recall:    {r:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  P-R-F1 Diff: {diff:.4f} [{status}]")

acc_f1_diff = abs(test_accuracy - macro_f1)
acc_status = "✓ PASS" if acc_f1_diff < 0.1 else "✗ FAIL"

print(f"\n{'='*70}")
print("Overall Metrics:")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Acc-F1 Diff:   {acc_f1_diff:.4f} [{acc_status}]")

print("\n" + "="*70)
if all_good and acc_f1_diff < 0.1:
    print("✓ ALL REQUIREMENTS MET!")
else:
    print("✗ Some requirements not met - may need further tuning")
print("="*70)

## 17. Save Model Information

In [ ]:
# Save class metrics
with open(os.path.join(MODEL_DIR, 'class_metrics.csv'), 'w') as f:
    f.write("Class,Precision,Recall,F1-Score,Support,P-R-F1 Diff,Status\n")
    for r in results:
        sup = report_dict[r['class']]['support']
        status = "PASS" if r['diff'] < 0.15 else "FAIL"
        f.write(f"{r['class']},{r['precision']:.4f},{r['recall']:.4f},{r['f1']:.4f},{sup},{r['diff']:.4f},{status}\n")

# Save model info
model_info = {
    'model_name': f'{MODEL_NAME}_{MODEL_VERSION}',
    'base_model': 'MobileNetV2',
    'version': MODEL_VERSION,
    'date_trained': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'input_size': [IMG_SIZE, IMG_SIZE, 3],
    'num_classes': num_classes,
    'classes': class_names,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'phase1_epochs': PHASE1_EPOCHS,
        'phase2_epochs': PHASE2_EPOCHS,
        'initial_lr': INITIAL_LR,
        'dropout_rate': DROPOUT_RATE,
        'label_smoothing': LABEL_SMOOTHING,
        'focal_gamma': FOCAL_GAMMA,
        'focal_alpha': FOCAL_ALPHA
    },
    'dataset': {
        'train_samples': train_generator.samples,
        'val_samples': val_generator.samples,
        'test_samples': test_generator.samples
    },
    'metrics': {
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss),
        'macro_precision': float(report_dict['macro avg']['precision']),
        'macro_recall': float(report_dict['macro avg']['recall']),
        'macro_f1': float(report_dict['macro avg']['f1-score'])
    },
    'class_metrics': {
        name: {
            'precision': float(report_dict[name]['precision']),
            'recall': float(report_dict[name]['recall']),
            'f1_score': float(report_dict[name]['f1-score']),
            'support': int(report_dict[name]['support'])
        }
        for name in class_names
    },
    'supervisor_requirements': {
        'p_r_f1_balanced': all_good,
        'acc_f1_close': acc_f1_diff < 0.1
    }
}

with open(os.path.join(MODEL_DIR, 'model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("="*70)
print("MODEL INFORMATION SAVED")
print("="*70)
print(f"Results saved to: {MODEL_DIR}")
print("\nFiles saved:")
for f in os.listdir(MODEL_DIR):
    print(f"  - {f}")

## 18. Final Summary

In [ ]:
print("="*70)
print("                    TRAINING COMPLETE!")
print("="*70)
print(f"\nModel: {MODEL_NAME}_{MODEL_VERSION}")
print(f"Base Model: MobileNetV2")
print(f"Model saved to: {MODEL_DIR}")

print(f"\n{'='*70}")
print("                    FINAL RESULTS")
print("="*70)
print(f"\n  Test Accuracy:    {test_accuracy*100:.2f}%")
print(f"  Macro F1-Score:   {macro_f1*100:.2f}%")
print(f"  Macro Precision:  {report_dict['macro avg']['precision']*100:.2f}%")
print(f"  Macro Recall:     {report_dict['macro avg']['recall']*100:.2f}%")

print(f"\n{'='*70}")
print("                 CLASS-WISE F1-SCORES")
print("="*70)
for name in class_names:
    f1 = report_dict[name]['f1-score']
    print(f"\n  {name}:" + " "*(20-len(name)) + f"{f1*100:.2f}%")

print(f"\n{'='*70}")
print("                    FILES GENERATED")
print("="*70)
print("\n  - best_model.keras")
print("  - phase1_model.keras")
print("  - model_info.json")
print("  - class_metrics.csv")
print("  - confusion_matrix.png")
print("  - training_history.png")
print("  - per_class_metrics.png")
print("  - sample_images.png")
print("  - dataset_distribution.png")
print("\n" + "="*70)